In [28]:
!pip install -q git+https://github.com/PyThaiNLP/pythainlp

In [29]:
# install pythainlp and ssg(subword tokenizer)
!pip install -q ssg

In [30]:
from functools import lru_cache
from pythainlp.tokenize import subword_tokenize, word_tokenize
from pythainlp.util import sound_syllable, Trie
from pythainlp.util import remove_tonemark
from pythainlp.khavee import KhaveeVerifier
import pythainlp as pythai
from pythainlp.util import isthai
from pythainlp.transliterate import pronunciate
from pythainlp.spell import correct
from tqdm import tqdm
import numpy as np
import pandas as pd
kv = KhaveeVerifier()

### Word and Subword Tokenizing

In [31]:
# split text from \n to list and drop soi word ->  splitted wak list (no soi)
def split_klong(klong_text):
  splitted_klong = []
  klong_list = klong_text.split('\n')
  klong_list = [klong for klong in klong_list if klong.strip()]
  for i in range(len(klong_list)):
    if i == 1 or i == 3 or i == 5:
      klong = klong_list[i]
      if klong[0] == ' ':
        klong = klong[1:]
      klong = klong.split(' ')
      print(f"klong: {klong}")
      splitted_klong.append(klong[0])
    else:
      splitted_klong.append(klong_list[i].replace(' ', ''))
  return splitted_klong

In [32]:
# subword tokenize wak with ssg and dict
def subword_token(wak, engine='ssg'):
  subword_tokenized = subword_tokenize(wak, engine='ssg')
  if len(subword_tokenized) != 5 and len(subword_tokenized) != 2:
      subword_tokenized = subword_tokenize(wak, engine='dict')
  return subword_tokenized

In [33]:
klong_txt = """เสียงลือเสียงเล่าอ้าง
อันใด พี่เอย
เสียงย่อมยอยศใคร
ทั่วหล้า
สองเขือพี่หลับไหล
ลืมตื่น ฤๅพี่
สองพี่คิดเองอ้า
อย่าได้ถามเผือ"""
klong_txt2 = """พระสมุทรสุดลึกล้น
คณนา
สายดิ่งทิ้งทอดมา
หยั่งได้
เขาสูงอาจวัดวา
กำหนด
จิตมนุษย์นี้ไซร้
ยากแท้หยั่งถึง
"""
print(klong_txt2+"\n\n")
splitted_klong = split_klong(klong_txt2)
print(f"splitted_klong: {splitted_klong}\n\n")
for i in range(len(splitted_klong)):
  wak = splitted_klong[i]
  subword_tokenized = subword_token(wak)
  print(f"wak: {wak} -> subword tokenized: {subword_tokenized}")

พระสมุทรสุดลึกล้น
คณนา
สายดิ่งทิ้งทอดมา
หยั่งได้
เขาสูงอาจวัดวา
กำหนด
จิตมนุษย์นี้ไซร้
ยากแท้หยั่งถึง



klong: ['คณนา']
klong: ['หยั่งได้']
klong: ['กำหนด']
splitted_klong: ['พระสมุทรสุดลึกล้น', 'คณนา', 'สายดิ่งทิ้งทอดมา', 'หยั่งได้', 'เขาสูงอาจวัดวา', 'กำหนด', 'จิตมนุษย์นี้ไซร้', 'ยากแท้หยั่งถึง']


wak: พระสมุทรสุดลึกล้น -> subword tokenized: ['พระ', 'สมุทร', 'สุด', 'ลึก', 'ล้น']
wak: คณนา -> subword tokenized: ['คณ', 'นา']
wak: สายดิ่งทิ้งทอดมา -> subword tokenized: ['สาย', 'ดิ่ง', 'ทิ้ง', 'ทอด', 'มา']
wak: หยั่งได้ -> subword tokenized: ['หยั่ง', 'ได้']
wak: เขาสูงอาจวัดวา -> subword tokenized: ['เขา', 'สูง', 'อาจ', 'วัด', 'วา']
wak: กำหนด -> subword tokenized: ['กำ', 'หนด']
wak: จิตมนุษย์นี้ไซร้ -> subword tokenized: ['จิต', 'มนุษย์', 'นี้', 'ไซ', 'ร้']
wak: ยากแท้หยั่งถึง -> subword tokenized: ['ยาก', 'แท้', 'หยั่ง', 'ถึง']


### Check Functions

#### Number of syllables check

In [34]:

# check number of syllables -> [True, True, True, True, True, True, True, True] (len=8)
def subword_num(splitted_klong):
  checked = []
  two = [1,3,5]
  five = [0,2,4,6]
  for num in range(len(splitted_klong)):
    if num in two:
      checked.append(len(subword_token(splitted_klong[num])) == 2)
    elif num in five:
      checked.append(len(subword_token(splitted_klong[num])) == 5)
    elif num == 7:
      checked.append(len(subword_token(splitted_klong[num])) == 4)
  return checked

#### eak tou check


In [35]:
# check what word tone is
def find_tone(word):
  char_list = [*word]
  if "่" in char_list or sound_syllable(word) == 'dead':
    return "eak or dead"
  elif "้" in char_list:
    return "tou"
  else:
    return False

In [36]:
# check eaktou -> list[True, True, True, True, True, True, True, True] (len=8)
def check_eaktou(splitted_klong):
  checked = []
  for num in range(len(splitted_klong)):
    tokenzied_wak = subword_token(splitted_klong[num])
    if num == 0:
      checked.append(find_tone(tokenzied_wak[3]) == "eak or dead" and find_tone(tokenzied_wak[4]) == 'tou')
    elif num == 1:
      checked.append(True)
    elif num == 2:
      checked.append(find_tone(tokenzied_wak[1]) == "eak or dead")
    elif num == 3:
      checked.append(find_tone(tokenzied_wak[0]) == 'eak or dead' and find_tone(tokenzied_wak[1]) == 'tou')
    elif num == 4:
      checked.append(find_tone(tokenzied_wak[2]) == 'eak or dead')
    elif num == 5:
      checked.append(find_tone(tokenzied_wak[1]) == 'eak or dead')
    elif num == 6:
      checked.append(find_tone(tokenzied_wak[1]) == "eak or dead" and find_tone(tokenzied_wak[4]) == 'tou')
    elif num == 7:
      checked.append(find_tone(tokenzied_wak[0]) == "eak or dead" and find_tone(tokenzied_wak[1]) == 'tou')
  return checked

#### sampas check

In [37]:
# last sound of wak from pronunciate tokenized last word of each wak
# ex [เสียงลือเสียงเล่าอ้าง] -> [อ้าง]
def sound_words(splitted_klong):
  sound_list = []
  for wak in splitted_klong:
    list_char = [*wak]
    if " " in list_char:
      wak = wak.split(" ")
      wak = wak[0]
    wak = word_tokenize(wak, engine="newmm")
    pronounce_word = pronunciate(wak[-1], engine="w2p")
    sound_list.append(pronounce_word.replace('ฺ', '').split('-')[-1])
  return sound_list

In [38]:
# check sampas -> [True, True, True]
# [0] = sampas wak 2-3, [1] = sampas wak 2-4, [2] sampas wak 4-7
def check_sampas(sound_list):
  checked = []
  if len(sound_list) > 2:
    checked.append(kv.check_sumpus(sound_list[1],sound_list[2]))
    if len(sound_list) > 4:
      checked.append(kv.check_sumpus(sound_list[1],sound_list[4]))
      if len(sound_list) > 6:
        checked.append(kv.check_sumpus(sound_list[3],sound_list[6]))
  else:
    checked.append(True)
  return checked

#### Main Check

In [39]:
def main_check(klong_text):
  splitted_klong = split_klong(klong_text)
  checked_subword_num = subword_num(splitted_klong)
  if False in checked_subword_num:
    false_index = checked_subword_num.index(False)
    return 'syllable format error', false_index+1
  else:
    checked_eaktou = check_eaktou(splitted_klong)
    if False in checked_eaktou:
      false_index = checked_eaktou.index(False)
      return 'eaktou format error', false_index+1
    else:
      sound_list = sound_words(splitted_klong)
      checked_sampas = check_sampas(sound_list)
      if False in checked_sampas:
        wak_sampas = ['2 and 3', '2 and 5', '4 and 7']
        return 'sampas format error', wak_sampas[checked_sampas.index(False)]
      else:
        return True

In [40]:
def analyze_wak(wak_text):
    # 1. Tokenize into words
    words = word_tokenize(wak_text, engine="newmm")
    
    wak_data = []
    total_syllables = 0
    
    # 2. Analyze each word
    for word in words:
        # Ignore whitespace
        if word.strip() == "":
            continue
            
        spoken = pronunciate(word, engine="w2p")
        syllables = spoken.split("-")
        count = len(syllables)
        
        wak_data.append({
            "original_word": word,
            "spoken_form": spoken,
            "syllable_count": count
        })
        
        total_syllables += count

    # 3. Classify the Wak based on your rules
    classification = ""
    target_rhythm = []
    needs_manual_review = False
    
    if total_syllables == 9:
        classification = "9 Syllables"
        target_rhythm = [3, 3, 3]
    elif total_syllables == 8:
        classification = "8 Syllables"
        target_rhythm = [3, 2, 3]
    elif total_syllables == 7:
        classification = "7 Syllables"
        # Flagging for manual review as requested
        target_rhythm = [3, 2, 2] # Default assumption
        needs_manual_review = True
    else:
        classification = f"Irregular ({total_syllables} Syllables)"
        needs_manual_review = True

    return {
        "total_syllables": total_syllables,
        "classification": classification,
        "rhythm": target_rhythm,
        "needs_review": needs_manual_review,
        "word_breakdown": wak_data
    }

# --- Example Usage ---
test_wak = "ธรรมชาติสวยงามตามภูผา" # ธรรมชาติ (3) สวยงาม (2) ตาม (1) ภูผา (2) = 8
test_wak2 = "ปรารถนากนิษฐภคินี"
test_wak3 = "อุปการคุณกเฬวรากสมรรถภาพกิตติมศักดิ์ปรากฏการณ์มัธยัสถ์ภูมิลำเนามนุษย์อินทคาม ก ธ ร ฤ ลืมตื่นฤๅพี่"
test_wak4 = "อัธยาศัยโปร่งใสไสยศาสตร์วิสัยสดใสสงสัย"
result = analyze_wak(test_wak)
result2 = analyze_wak(test_wak2)
result3 = analyze_wak(test_wak3)
result4 = analyze_wak(test_wak4)

print(f"Total: {result['total_syllables']} ({result['classification']})")
print(f"Rhythm: {result['rhythm']}")
print(f"Review Needed: {result['needs_review']}")
for item in result['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")
    
print(f"Total: {result2['total_syllables']} ({result2['classification']})")
print(f"Rhythm: {result2['rhythm']}")
print(f"Review Needed: {result2['needs_review']}")
for item in result2['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")

print(f"Total: {result3['total_syllables']} ({result3['classification']})")
print(f"Rhythm: {result3['rhythm']}")
print(f"Review Needed: {result3['needs_review']}")
for item in result3['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")
    
print(f"Total: {result4['total_syllables']} ({result4['classification']})")
print(f"Rhythm: {result4['rhythm']}")
print(f"Review Needed: {result4['needs_review']}")
for item in result4['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")

Total: 8 (8 Syllables)
Rhythm: [3, 2, 3]
Review Needed: False
 - ธรรมชาติ -> ทำ-มะ-ชาด (3 beats)
 - สวยงาม -> สวย-งาม (2 beats)
 - ตาม -> ตาม (1 beats)
 - ภูผา -> พู-ผา (2 beats)
Total: 9 (9 Syllables)
Rhythm: [3, 3, 3]
Review Needed: False
 - ปรารถนา -> ปราด-ถะ-หนา (3 beats)
 - กนิษฐภคินี -> กะ-นิด-ถะ-พะ-คะ-นี (6 beats)
Total: 41 (Irregular (41 Syllables))
Rhythm: []
Review Needed: True
 - อุปการคุณ -> อุบ-ปะ-กาน-คุน (4 beats)
 - กเฬวราก -> กะ-เล-วะ-ราก (4 beats)
 - สมรรถภาพ -> สะ-มะ-ถะ-พาบ (4 beats)
 - กิตติมศักดิ์ -> กิด-ติม-สัก (3 beats)
 - ปรากฏการณ์ -> ปฺรา-กด-กาน (3 beats)
 - มัธยัสถ์ -> มัด-ทะ-ยัด (3 beats)
 - ภูมิลำเนา -> พูม-ลำ-เนา (3 beats)
 - มนุษย์ -> มะ-นุด (2 beats)
 - อิน -> อิน (1 beats)
 - ท -> ทอด (1 beats)
 - คาม -> คาม (1 beats)
 - ก -> กะ-โหฺมด (2 beats)
 - ธ -> ทอน (1 beats)
 - ร -> ระ-คะ-แนน (3 beats)
 - ฤ -> รึ-เริฟ (2 beats)
 - ลืม -> ลืม (1 beats)
 - ตื่น -> ตื่น (1 beats)
 - ฤๅ -> รือ (1 beats)
 - พี่ -> พี่ (1 beats)
Total: 14 (Irregular (14 Syllables))
Rhy

In [41]:
# pronunciate can't be used with 1-2 characters, as it would hallucinate the pronunciation.
# problematic_word = ["ฤๅ","ก" ,"ข", "ค", "ฆ", "ง", "จ", "ฉ", "ช", "ซ", "ฌ", "ญ", "ฎ", "ฏ", "ฐ", "ฑ", "ฒ", "ณ", "ด", "ต", "ถ", "ท", "ธ", "น", "บ", "ป", "ผ", "ฝ", "พ", "ฟ", "ภ", "ม", "ย", "ร", "ล", "ว", "ฤ"]
# for word in problematic_word:
#     result = pronunciate(word, engine="w2p")
#     print(f"{word} -> {result}")

In [42]:
# `kv.check_klon` is a method from the KhaveeVerifier (Not that accurate)
print(kv.check_klon(
    "แม่วันทองของลูกจงกลับบ้าน ขาจะพาลว้าวุ่นแม่ทูนหัว จะก้มหน้าลาไปมิได้กลัว แม่อย่ามัวหมองนักจงหักใจ",
    k_type=8
))

The poem is correct according to the principle.


In [43]:
print(kv.is_sumpus("บ้าน", "พาล"))
print(kv.is_sumpus("กัย", "กัย"))

True
True


In [44]:
print(kv.check_sara("พาล"))
print(kv.check_marttra("พาล"))
print(kv.check_marttra("พาน"))

อา
กน
กน


In [45]:
# `kv._has_true_final_yl` is deprecated the new method is `kv._is_true_final`.
# Currerntly, `kv._has_true_final_yl` is pointed to `kv._is_true_final`
# print(kv._has_true_final_yl("ล"))

In [46]:
from pythainlp.tokenize import word_tokenize, syllable_tokenize
from pythainlp.transliterate import pronunciate

In [ ]:
# === GOLD STANDARD LOOKUP DICTIONARY (moved to poetry_overrides.py) ===
# The big dict now lives in poetry_overrides.py so the notebook, the cleaner,
# and build_overrides.py all share ONE source of truth. Edit the module, then
# re-run this cell to reload it.
from poetry_overrides import POETRY_OVERRIDES
print(f"POETRY_OVERRIDES loaded: {len(POETRY_OVERRIDES)} entries")


In [48]:
# Build COMBINED Trie: default newmm dictionary + our poetic vocabulary
# custom_dict REPLACES (not augments) the default dict, so we must merge them.
from pythainlp.tokenize import word_dict_trie

_poetry_trie = word_dict_trie()  # start with newmm's built-in dictionary
_added = 0
for k in POETRY_OVERRIDES:
    if len(k) > 1 and k not in _poetry_trie:
        _poetry_trie.add(k)
        _added += 1
print(f"Combined Trie: default dict + {_added} new poetic entries (e.g. วิญญาณ์, ศอก, ปรวาทยาจารย์)")


Combined Trie: default dict + 0 new poetic entries (e.g. วิญญาณ์, ศอก, ปรวาทยาจารย์)


In [49]:
print(f"Combined Trie size: {_poetry_trie._word_count} entries")
print(f"first 50 entries: {list(_poetry_trie)[:50]}")

Combined Trie size: 62116 entries
first 50 entries: ['คุ', 'คุณ', 'คุณค่า', 'คุณความดี', 'คุณครู', 'คุณหนู', 'คุณหมอ', 'คุณหญิง', 'คุณภาพ', 'คุณภาพดี', 'คุณภาพต่ำ', 'คุณภาพประชากร', 'คุณภาพชีวิต', 'คุณภาพสิ่งแวดล้อม', 'คุณย่า', 'คุณยาย', 'คุณพ่อ', 'คุณพ่ออธิการ', 'คุณพระศรีรัตนตรัย', 'คุณพระช่วย', 'คุณพิเศษ', 'คุณประโยชน์', 'คุณปู่', 'คุณป้า', 'คุณนาม', 'คุณนาย', 'คุณลักษณะ', 'คุณลุงคุณป้า', 'คุณวิเศษ', 'คุณวุฒิ', 'คุณๆ', 'คุณธรรม', 'คุณผู้หญิง', 'คุณผู้ชาย', 'คุณากร', 'คุณานุประโยค', 'คุณตา', 'คุณสมบัติ', 'คุณสมบัติพิเศษ', 'คุณศัพท์', 'คุณูปการ', 'คุณไสย', 'คุณแม่', 'คุณชาย', 'คุณบท', 'คุณงามความดี', 'คุณอา', 'คุ้ม', 'คุ้มดีคุ้มร้าย', 'คุ้มค่า']


In [50]:
# # === TRIE & CUSTOM DICT PLAYGROUND ===
# # A Trie (Prefix Tree) is a tree where each node is a character.
# # Following a path root -> leaf spells out a dictionary word.
# # newmm walks the Trie at each position, picking the longest match.

# from pythainlp.util import Trie
# from pythainlp.tokenize import word_tokenize, word_dict_trie

# # --- 1. What's in a Trie? ---
# tiny = Trie(["ศอก", "วาง", "กวาง", "ศอ"])
# print("=== Trie membership (O(len(word)) lookup) ===")
# for w in ["ศอก", "ศอ", "ศ", "กวาง", "ศอกวาง"]:
#     print(f"  '{w}' in Trie: {w in tiny}")

# # --- 2. Tokenization WITHOUT custom dict ---
# text = "กระบองสี่ศอกวางไว้ข้างกาย"
# without = word_tokenize(text, engine="newmm")
# print("\n=== WITHOUT custom dict (default newmm) ===")
# print(f"  Tokens: {without}")
# print("  -> 'ศอก' not in built-in dict, so newmm splits: ศอ + กวาง")

# # --- 3. Tokenization WITH tiny Trie that REPLACES default ---
# # WARNING: custom_dict REPLACES (not augments) the default dict!
# with_tiny = word_tokenize(text, engine="newmm", custom_dict=tiny)
# print("\n=== WITH tiny Trie (4 words, REPLACES default) ===")
# print(f"  Tokens: {with_tiny}")
# print("  -> Only 4 words known; everything else gets garbled")

# # --- 4. Tokenization with COMBINED Trie (default + our words) ---
# combined = word_dict_trie()        # start with newmm's full dictionary
# combined.add("ศอก")               # teach it one new word
# with_combined = word_tokenize(text, engine="newmm", custom_dict=combined)
# print("\n=== WITH combined Trie (default + 'ศอก') ===")
# print(f"  Tokens: {with_combined}")
# print("  -> 'ศอก' recognised, AND common words still work!")

# # --- 5. How newmm vs longest resolve ambiguity differently ---
# print("\n=== newmm vs longest: same combined Trie, different algorithms ===")
# print(f"  Input: 'กระบองสี่ศอกวางไว้ข้างกาย'")
# tokens_nm = word_tokenize(text, engine="newmm", custom_dict=combined)
# tokens_lo = word_tokenize(text, engine="longest", custom_dict=combined)
# print(f"  newmm  : {tokens_nm}")
# print(f"  longest: {tokens_lo}")
# print()
# print("  Key difference:")
# print("    newmm   = graph-based, uses corpus edge-weights -> ศอ+กวาง wins on frequency")
# print("    longest = pure dictionary, picks longest valid word -> ศอก (3 chars) > ศอ (2 chars)")
# print("But longest is too greedy and don't understand context so it stole characters from the next word")
# print("and garbled the rest of the sentence. So we use newmm for poetry tokenization.")

In [51]:
@lru_cache(maxsize=2048)
def process_w2p(word: str) -> tuple:
    """Helper: pronunciate -> clean -> ssg-split.  LRU-cached (w2p is the bottleneck)."""
    # pronunciate hallucination on very short words. Use syllable_tokenize instead.
    if len(word) <= 2:
        return tuple(syllable_tokenize(word, engine="ssg"))
        
    phonetic_word = pronunciate(word, engine="w2p")
    
    if not phonetic_word:
        return tuple(syllable_tokenize(word, engine="ssg"))
    
    # Clean up unwanted characters like Phinthu (-ฺ) and hyphens (-)
    clean_phonetic = phonetic_word.replace("ฺ", "").replace("-", "")
    
    # Using ssg (CRF segmenter) as it handles phonetic text better than dict
    phonetic_syllables = syllable_tokenize(clean_phonetic, engine="ssg")
    
    # Clean up stray 'ห' artifacts
    return tuple(s for s in phonetic_syllables if s != "ห" or word == "ห")

In [52]:
def extract_poetic_syllables(text: str, custom_trie=None) -> list:
    """Extracts phonetic syllables for Klon 8 verification.
    
    Args:
        text: Raw wak text
        custom_trie: Optional pythainlp.util.Trie of poetic vocabulary.
                     The Trie teaches newmm to recognise words like ศอก, วิญญาณ์
                     so they aren't mis-split (e.g. ศอกวาง -> ศอ+กวาง).
    """
    # Tokenize text into words using the newmm (greedy) engine.
    # custom_trie augments newmm's built-in dictionary with our poetic vocabulary.
    words = word_tokenize(text, engine="newmm", custom_dict=custom_trie or _poetry_trie)
    final_syllables = []
    
    i = 0
    while i < len(words):
        word = words[i]
        
        # --- MERGE CHECK: try joining consecutive tokens to match an override ---
        # Handles cases where newmm splits a word like วิญญาณ์ -> วิ + ญญาณ์
        # MAX_MERGE = max number of TOKENS (not syllables) any override spans when split
        merged = False
        MAX_MERGE = 6
        end = min(i + MAX_MERGE, len(words))
        for j in range(end, i, -1):  # try longest match first within window
            combined = ''.join(words[i:j])
            if combined in POETRY_OVERRIDES:
                final_syllables.extend(POETRY_OVERRIDES[combined])
                i = j
                merged = True
                break
        
        if merged:
            continue
        
        # --- STOLEN-CHAR CHECK: newmm sometimes steals a char into the next token ---
        # e.g. ศอกวาง -> tokenizer gives ['ศอ', 'กวาง']; 'ศอ' + 'ก' = 'ศอก' is an override.
        # Try merging word[i] with up to 3 leading chars of word[i+1].
        if i + 1 < len(words):
            next_word = words[i + 1]
            for k in range(1, min(4, len(next_word) + 1)):
                candidate = word + next_word[:k]
                if candidate in POETRY_OVERRIDES:
                    final_syllables.extend(POETRY_OVERRIDES[candidate])
                    # Replace the next token with the remainder, or remove it if consumed
                    remainder = next_word[k:]
                    if remainder:
                        words[i + 1] = remainder
                    else:
                        words.pop(i + 1)
                    i += 1
                    merged = True
                    break
            if merged:
                continue
        
        # Direct Override if the tokenized word is in the POETRY_OVERRIDES list (O(1) Fast Lookup)
        if word in POETRY_OVERRIDES:
            final_syllables.extend(POETRY_OVERRIDES[word])
            i += 1
            continue

        # Use 'ssg' to check if newmm greedy engine merged words like "ได้ใจ" or "รู้อยู่"
        sub_syllables = syllable_tokenize(word, engine="ssg")
        
        # Check if ANY of the segmented syllables are in the POETRY_OVERRIDES list
        has_override = any(sub in POETRY_OVERRIDES for sub in sub_syllables)
        
        if len(sub_syllables) > 1 and has_override:
            # Found a hidden override word in the segmented syllables. Process each sub-syllable independently.
            for sub in sub_syllables:
                if sub in POETRY_OVERRIDES:
                    final_syllables.extend(POETRY_OVERRIDES[sub])
                else:
                    final_syllables.extend(process_w2p(sub))
            i += 1
            continue
        
        # No overrides found -> treat as a true compound word (e.g. พัฒนาการ).
        # Fall back to w2p pronunciate engine.
        final_syllables.extend(process_w2p(word))
        i += 1
        
    return final_syllables

In [53]:
# # Test sentences
# sentence = [
#     "สรรเพชญโพธิญาณประมาณหมาย", 
#     # "โอ้มนุษย์ผู้มีจิตใจและมีความสามารถในการสร้างสรรค์สิ่งใหม่",
#     # "ความแปลกแยกและพัฒนาการ",
#     # "กับบรมบิตุเรศพระมารดา",
#     # "อันประกอบด้วยสระและพยัญชนะหลายตัว",
# ]

# for sent in sentence:
#     syllables = extract_poetic_syllables(sent)
#     print(f"Sentence: {sent}")
#     print(f"Final Syllable: {syllables}\n")

In [55]:
# Read phraAphai_1_export.csv and display syllable breakdown for each line
import pandas as pd

# NOTE: export files now live under Results/Exportable/ (not at the workspace root)
df = pd.read_csv('Results/Exportable/phraAphai_1_export.csv')

print(f"Total rows (stanzas): {len(df)}")
print("=" * 80)

for row_idx, row in df.iterrows():
    print(f"\n--- Stanza {row_idx + 1} ---")
    
    for wak_num in range(1, 9):
        # Combine the 3 parts (a, b, c) for this wak
        part_a = row[f'w{wak_num}_a']
        part_b = row[f'w{wak_num}_b']
        part_c = row[f'w{wak_num}_c']
        
        full_wak = f"{part_a}{part_b}{part_c}"
        syllables = extract_poetic_syllables(full_wak)
        
        print(f"\n  Wak {wak_num}: {full_wak}")
        print(f"  Syllables ({len(syllables)}): {syllables}")
    
    # Print separator between stanzas if not last
    if row_idx < len(df) - 1:
        print("\n" + "-" * 80)


Total rows (stanzas): 57

--- Stanza 1 ---

  Wak 1: สมเด็จท้าวบิตุรงค์ดำรงราชย์
  Syllables (9): ['สม', 'เด็ด', 'ท้าว', 'บิ', 'ตุ', 'รง', 'ดำ', 'รง', 'ราด']

  Wak 2: แสนสวาทลูกน้อยเสน่หา
  Syllables (8): ['แสน', 'สะ', 'หวาด', 'ลูก', 'น้อย', 'สะ', 'เหน่', 'หา']

  Wak 3: จะเสกสองครองสมบัติขัตติยา
  Syllables (9): ['จะ', 'เสก', 'สอง', 'ครอง', 'สม', 'บัด', 'ขัด', 'ติ', 'ยา']

  Wak 4: แต่วิชาสิ่งใดไม่ชำนาญ
  Syllables (8): ['แต่', 'วิ', 'ชา', 'สิ่ง', 'ได', 'ไม่', 'ชำ', 'นาน']

  Wak 5: จึงดำรัสเรียกพระโอรสราช
  Syllables (8): ['จึง', 'ดำ', 'รัด', 'เรียก', 'พระ', 'โอ', 'รด', 'ราด']

  Wak 6: มาริมอาสน์แท่นสุวรรณแล้วบรรหาร
  Syllables (9): ['มา', 'ริม', 'อาด', 'แท่น', 'สุ', 'วัน', 'แล้ว', 'บัน', 'หาร']

  Wak 7: พ่อจะแจ้งเจ้าจงจำคำโบราณ
  Syllables (9): ['พ่อ', 'จะ', 'แจ้ง', 'เจ้า', 'จง', 'จำ', 'คำ', 'โบ', 'ราน']

  Wak 8: อันชายชาญเชื้อกษัตริย์ขัตติยา
  Syllables (9): ['อัน', 'ชาย', 'ชาน', 'เชื้อ', 'กะ', 'สัด', 'ขัด', 'ติ', 'ยา']

---------------------------------------------------------

In [56]:
# === W2P DIAGNOSTIC: show every word that falls through to process_w2p ===
# Group by original word, show w2p output and frequency.
# Use this to spot w2p errors -> add overrides to POETRY_OVERRIDES.

from collections import defaultdict

w2p_log = defaultdict(lambda: {"syllables": None, "count": 0, "first_wak": None})

for row_idx, row in df.iterrows():
    for wak_num in range(1, 9):
        full_wak = f"{row[f'w{wak_num}_a']}{row[f'w{wak_num}_b']}{row[f'w{wak_num}_c']}"
        # Tokenize the same way extract_poetic_syllables does
        tokens = word_tokenize(full_wak, engine="newmm", custom_dict=_poetry_trie)
        for token in tokens:
            if token in POETRY_OVERRIDES:
                continue  # handled by override, skip
            # This token would fall through to process_w2p
            syl = process_w2p(token)
            key = token
            if w2p_log[key]["syllables"] is None:
                w2p_log[key]["syllables"] = list(syl)
                w2p_log[key]["first_wak"] = full_wak[:40]
            w2p_log[key]["count"] += 1

# Sort by frequency (descending), show top 60
sorted_words = sorted(w2p_log.items(), key=lambda x: -x[1]["count"])

print(f"{'Word':<16} {'Freq':>4}  {'w2p syllables':<28}  Example context")
print("-" * 90)
for word, info in sorted_words[:60]:
    syl_str = "-".join(info["syllables"])
    ctx = info["first_wak"] or ""
    print(f"{word:<16} {info['count']:>4}  {syl_str:<28}  {ctx}")

# Also show words with 3+ syllables (likely candidates for overrides)
print(f"\n{'='*90}")
print("Multi-syllable words (3+ syllables) — good override candidates:")
print(f"{'='*90}")
for word, info in sorted_words:
    if len(info["syllables"]) >= 3:
        syl_str = "-".join(info["syllables"])
        print(f"  {word:<16} -> [{', '.join(repr(s) for s in info['syllables'])}]  (x{info['count']})")

Word             Freq  w2p syllables                 Example context
------------------------------------------------------------------------------------------
คน                 24  คน                            มีทิศาปาโมกข์อยู่สองคน
สอง                21  สอง                           จะเสกสองครองสมบัติขัตติยา
วิชา               18  วิ-ชา                         แต่วิชาสิ่งใดไม่ชำนาญ
พราหมณ์            15  พราม                          เรียกว่าบ้านจันตคามพราหมณ์พฤฒา
แสน                14  แสน                           แสนสวาทลูกน้อยเสน่หา
กษัตริย์           14  กะ-สัด                        อันชายชาญเชื้อกษัตริย์ขัตติยา
นั้น               14  นั้น                          อันทองแสนตำลึงนั้นไม่ทันหา
อัน                13  อัน                           อันชายชาญเชื้อกษัตริย์ขัตติยา
บ้าน               13  บ้าน                          ล่วงตำบลชนบทไปหลายบ้าน
น้อง               13  น้อง                          พระพี่เก็บกาหลงส่งให้น้อง
อาจารย์            13  อา-จาน                      

In [ ]:
# # === TIMING: old vs new extract_poetic_syllables ===
# # NOTE: lru_cache on process_w2p makes repeated runs faster (cache warming).
# # We clear the cache before EACH timing run for a fair comparison.
# import time
# import pandas as pd

# # --- OLD version (no merge check, no stolen-char, no custom trie) ---
# def extract_poetic_syllables_OLD(text: str) -> list:
#     words = word_tokenize(text, engine="newmm")
#     final_syllables = []
#     for word in words:
#         if word in POETRY_OVERRIDES:
#             final_syllables.extend(POETRY_OVERRIDES[word])
#             continue
#         sub_syllables = syllable_tokenize(word, engine="ssg")
#         has_override = any(sub in POETRY_OVERRIDES for sub in sub_syllables)
#         if len(sub_syllables) > 1 and has_override:
#             for sub in sub_syllables:
#                 if sub in POETRY_OVERRIDES:
#                     final_syllables.extend(POETRY_OVERRIDES[sub])
#                 else:
#                     final_syllables.extend(process_w2p(sub))
#             continue
#         final_syllables.extend(process_w2p(word))
#     return final_syllables

# df = pd.read_csv('phraAphai_1_export.csv')

# # Collect all wak texts
# all_waks = []
# for row_idx, row in df.iterrows():
#     for wak_num in range(1, 9):
#         full_wak = f"{row[f'w{wak_num}_a']}{row[f'w{wak_num}_b']}{row[f'w{wak_num}_c']}"
#         all_waks.append(full_wak)

# N_WAKS = len(all_waks)
# print(f"Total waks: {N_WAKS} (across {len(df)} stanzas)")

# # Time OLD version (cold cache)
# process_w2p.cache_clear()
# start = time.perf_counter()
# for wak in all_waks:
#     extract_poetic_syllables_OLD(wak)
# time_old = time.perf_counter() - start
# print(f"\nOLD (no merge, cold cache): {time_old:.4f}s  ({time_old/N_WAKS*1000:.2f} ms/wak)")

# # Time NEW version (cold cache)
# process_w2p.cache_clear()
# start = time.perf_counter()
# for wak in all_waks:
#     extract_poetic_syllables(wak)
# time_new = time.perf_counter() - start
# print(f"NEW (merge + stolen-char, cold cache): {time_new:.4f}s  ({time_new/N_WAKS*1000:.2f} ms/wak)")

# overhead = (time_new - time_old) / time_old * 100 if time_old > 0 else 0
# print(f"\nOverhead: {overhead:+.1f}%")

# # Estimate for 1000 stanzas (8000 waks)
# est_1000 = time_new / N_WAKS * 8000
# print(f"Estimated for 1000 stanzas: {est_1000:.2f}s ({est_1000/60:.1f} min)")

# # Warm-cache estimate (second run is always faster due to lru_cache)
# process_w2p.cache_clear()
# for wak in all_waks:
#     extract_poetic_syllables(wak)  # warm up
# start = time.perf_counter()
# for wak in all_waks:
#     extract_poetic_syllables(wak)
# time_warm = time.perf_counter() - start
# print(f"NEW (warm cache, 2nd run): {time_warm:.4f}s  ({time_warm/N_WAKS*1000:.2f} ms/wak)")
# est_1000_warm = time_warm / N_WAKS * 8000
# print(f"Estimated for 1000 stanzas (warm): {est_1000_warm:.2f}s ({est_1000_warm/60:.1f} min)")

### Full Corpus: Combine All Export Files (phraAphai_1 .. phraAphai_132)

The earlier cells only read `phraAphai_1_export.csv`. Here we merge all 132
chapter exports into a single `df_all`, keeping a `chapter` column so we can
trace any word back to its source.


In [57]:
# === 1. LOAD & COMBINE ALL 132 EXPORT FILES ===
import glob, os, re

EXPORT_DIR = 'Results/Exportable'

def chapter_num(path):
    """Extract the chapter number from a filename like phraAphai_101_export.csv -> 101"""
    return int(re.search(r'(\d+)', os.path.basename(path)).group(1))

# Sort numerically (1, 2, ..., 132), NOT alphabetically (1, 10, 100, ...)
files = sorted(glob.glob(os.path.join(EXPORT_DIR, 'phraAphai_*_export.csv')), key=chapter_num)
print(f"Found {len(files)} export files: chapter {chapter_num(files[0])} .. {chapter_num(files[-1])}")

frames = []
for f in files:
    d = pd.read_csv(f)
    d.insert(0, 'chapter', chapter_num(f))          # which of the 132 chapters (ผูก)
    d.insert(1, 'stanza_no', range(1, len(d) + 1))  # stanza index within that chapter
    frames.append(d)

df_all = pd.concat(frames, ignore_index=True)
print(f"df_all shape: {df_all.shape}  ({len(df_all):,} stanzas total)")
print(f"Chapters: {df_all['chapter'].min()}..{df_all['chapter'].max()} ({df_all['chapter'].nunique()} unique)")
print(f"Stanzas per chapter: min={df_all.groupby('chapter').size().min()}, "
      f"max={df_all.groupby('chapter').size().max()}, mean={df_all.groupby('chapter').size().mean():.1f}")

# Sanity: every chapter must have exactly the 24 wak columns
expected_cols = [f'w{n}_{p}' for n in range(1, 9) for p in ('a', 'b', 'c')]
missing = [c for c in expected_cols if c not in df_all.columns]
print(f"Missing columns: {missing if missing else 'none'}")
print(df_all.head(2).to_string())


Found 132 export files: chapter 1 .. 132
df_all shape: (8481, 26)  (8,481 stanzas total)
Chapters: 1..132 (132 unique)
Stanzas per chapter: min=17, max=154, mean=64.2
Missing columns: none
   chapter  stanza_no         w1_a      w1_b       w1_c         w2_a     w2_b         w2_c       w3_a        w3_b     w3_c       w4_a    w4_b      w4_c      w5_a      w5_b        w5_c        w6_a        w6_b        w6_c         w7_a      w7_b       w7_c       w8_a           w8_b     w8_c
0        1          1   สมเด็จท้าว  บิตุรงค์  ดำรงราชย์      แสนสวาท  ลูกน้อย       เสน่หา   จะเสกสอง  ครองสมบัติ  ขัตติยา    แต่วิชา  สิ่งใด  ไม่ชำนาญ  จึงดำรัส  เรียกพระ     โอรสราช  มาริมอาสน์  แท่นสุวรรณ  แล้วบรรหาร    พ่อจะแจ้ง  เจ้าจงจำ    คำโบราณ  อันชายชาญ  เชื้อกษัตริย์  ขัตติยา
1        1          2  แล้วก้มกราบ   บิตุราช   มาตุรงค์  ทั้งสององค์  ลูบหลัง  แล้วสั่งสอน  จะเดินทาง     กลางป่า    พนาดร  จงผันผ่อน  ตรึกจำ   คำโบราณ   จะพูดจา    สารพัด  บำหยัดยั้ง   จนลุกนั่ง      น้ำท่า    กระยาหาร  แม้นหลับนอน 

In [ ]:
# === 2. FULL-CORPUS W2P DIAGNOSTIC (collect) ===
# Same logic as the chapter-1-only cell, but over ALL stanzas in df_all.
# For every token that is NOT handled by POETRY_OVERRIDES, record:
#   - its w2p syllable split  (this is the thing we want to get right / speed up)
#   - how many times it occurs
#   - an example wak it appeared in
# Re-run this cell after adding overrides -> covered% should climb.
from collections import defaultdict
from tqdm import tqdm

# Flat list of the 24 wak-part columns: w1_a, w1_b, w1_c, w2_a, ... w8_c
WAK_COLS = [f'w{n}_{p}' for n in range(1, 9) for p in ('a', 'b', 'c')]

def wak_text(row):
    """Rebuild a full wak from its 3 CSV parts, treating NaN as empty string."""
    return "".join(row[c] if isinstance(row[c], str) else "" for c in WAK_COLS)

all_waks = [wak_text(row) for _, row in df_all.iterrows()]
print(f"Total waks across all chapters: {len(all_waks):,}")

w2p_log = defaultdict(lambda: {"syllables": None, "count": 0, "first_wak": None})
total_tokens = 0
covered_tokens = 0

for wak in tqdm(all_waks, desc="tokenizing waks"):
    tokens = word_tokenize(wak, engine="newmm", custom_dict=_poetry_trie)
    total_tokens += len(tokens)
    for token in tokens:
        if token in POETRY_OVERRIDES:
            covered_tokens += 1
            continue
        info = w2p_log[token]
        if info["syllables"] is None:
            info["syllables"] = list(process_w2p(token))
            info["first_wak"] = wak[:40]
        info["count"] += 1

# Sorted view used by every following cell (freq desc)
sorted_words = sorted(w2p_log.items(), key=lambda x: -x[1]["count"])

n_fall = total_tokens - covered_tokens
print(f"\nTotal tokens      : {total_tokens:,}")
print(f"Handled by overrides: {covered_tokens:,} ({covered_tokens/total_tokens*100:.1f}%)")
print(f"Fall through to w2p: {n_fall:,} occurrences / {len(w2p_log):,} unique words")


Total waks across all chapters: 8,481


tokenizing waks: 100%|██████████| 8481/8481 [00:46<00:00, 183.06it/s]


Total tokens      : 427,457
Handled by overrides: 108,219 (25.3%)
Fall through to w2p: 319,238 occurrences / 11,604 unique words


In [ ]:
print(f"\n... ({len(sorted_words) - 100} more unique words below freq #{sorted_words[99][1]['count'] if len(sorted_words) >= 100 else 0})")


In [ ]:
# === 4. GOOD OVERRIDE CANDIDATES (heuristics) ===
# A word is a "good candidate" if it's frequent AND/OR w2p is likely wrong.
# Heuristics used to flag a word:
#   multi(3+)     -> 3+ syllables (worth a dict entry: perf + accuracy)
#   Pali-cluster  -> consonant + ร/ล (กร ปร พร คร ตร กล พล...), classic w2p hallucination
#   ฤ/ฦ          -> contains a rare Thai vowel w2p often botches
#   silent-์      -> ends with thanthakhat (์), silent letter w2p may over-read
#   w2p≠ssg(N)    -> w2p syllable count disagrees with ssg (N) -> needs a human eye
import re
from pythainlp.tokenize import syllable_tokenize

PALI_CLUSTER = re.compile(r'[ก-ฮ][รล]')            # e.g. ปร, กร, พล, คร
RARE_VOWEL   = re.compile(r'[ฤฦ]')
SILENT_FINAL = re.compile(r'[ก-ฮ][่-๋]?์$')         # ends with ์ (thanthakhat)

def flag_candidate(word, info):
    flags = []
    syls = info["syllables"]
    if len(syls) >= 3:
        flags.append(f"multi({len(syls)})")
    if len(word) >= 3 and PALI_CLUSTER.search(word):
        flags.append("Pali-cluster")
    if RARE_VOWEL.search(word):
        flags.append("ฤ/ฦ")
    if SILENT_FINAL.search(word):
        flags.append("silent-์")
    try:
        ssg = syllable_tokenize(word, engine="ssg")
        if len(ssg) != len(syls):
            flags.append(f"w2p≠ssg({len(ssg)})")
    except Exception:
        pass
    return flags

rows = []
for word, info in sorted_words:
    if info["count"] < 2:          # skip words that appear only once
        continue
    flags = flag_candidate(word, info)
    if not flags:
        continue
    rows.append((word, info["count"], "-".join(info["syllables"]), "+".join(flags)))

# Rank: more flags first, then higher frequency
rows.sort(key=lambda r: (-len(r[3].split('+')), -r[1]))

print(f"{'Word':<18} {'Freq':>5}  {'w2p syllables':<28}  Flags")
print("-" * 95)
for word, freq, syl, flags in rows[:120]:
    print(f"{word:<18} {freq:>5}  {syl:<28}  {flags}")

print(f"\nTotal flagged candidates: {len(rows)}")


In [ ]:
# === 5. GENERATE OVERRIDES — via the standalone pipeline (build_overrides.py) ===
# NOTE: this cell is now a thin wrapper. The real pipeline lives in
# build_overrides.py and is STANDALONE (reads everything from disk — no stale
# kernel state). It has 4 independent hallucination guards:
#   1. GOLD input    : override_draft_tierA_safe copy.py is read as GOLD and
#                      NEVER regenerated — your hand fixes win.
#   2. PREFER-ORTHOGRAPHY: keep the ssg spelling when sound-equivalent
#                      (ใช้ not ไช้, ผู้ not พู่, สาคร->สา-คร) so เอก/โท tone
#                      marks match the original orthography.
#   3. COMPOSITION   : compounds built from covered parts (น้ำตา = น้ำ + ตา)
#                      — no w2p call, no hallucination possible.
#   4. SCREENING     : per-syllable is_sumpus vs ssg + tone/fragment checks.
# Outputs:
#   poetry_overrides_generated.py         AUTO+FIXED+COMPOSED (safe to merge)
#   poetry_overrides_generated_review.py  fix by ear (incl. suspicious draft)
#   poetry_overrides_generated.screen.tsv full report w/ context
# SAFETY: the pipeline NEVER writes to poetry_overrides.py or your draft.
import subprocess, sys, os

def _find_root():
    d = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(d, "build_overrides.py")):
            return d
        nd = os.path.dirname(d)
        if nd == d:
            break
        d = nd
    return os.getcwd()

_root = _find_root()
print(f"[running build_overrides.py in {_root}]")
r = subprocess.run([sys.executable, "build_overrides.py"],
                   capture_output=True, text=True, cwd=_root)
out = r.stdout.splitlines()
print("\n".join(out[:44]))
if r.returncode != 0:
    print("ERROR:\n" + r.stderr[-3000:])
print("\n[REVIEW LIST from the .screen.tsv — fix by ear before merging]")
import io as _io
_tsv = os.path.join(_root, "poetry_overrides_generated.screen.tsv")
if os.path.exists(_tsv):
    _shown = 0
    for _line in _io.open(_tsv, encoding="utf-8"):
        _p = _line.rstrip("\n").split("\t")
        if len(_p) >= 5 and _p[4] == "REVIEW" and _p[3].strip():
            print(f"  {_p[0]:<18} {'-'.join([]) if False else _p[2]:<30} {_p[3]:<40} {_p[5] if len(_p) > 5 else ''}")
            _shown += 1
            if _shown >= 40:
                print("  ... (more in the .screen.tsv)")
                break
    print(f"({_shown}+ shown of the REVIEW bucket)")


In [ ]:
# === 5b. SUMMARY: how many entries in each tier + a peek ===
print(f"TIER A (SAFE, freq>=10): {len(tier_a)} words")
print(f"TIER B (REVIEW, freq>=2): {len(tier_b)} words")
print("\n--- TIER A peek (first 15) ---")
for w, line in tier_a[:15]:
    print(line)
print("\n--- TIER B peek (first 15) ---")
for w, line in tier_b[:15]:
    print(line)
